# **Telecom RAG Pipeline**


In [ ]:
!pip install pyngrok
!curl -s https://ngrok-agent.s3.amazonaws.com/ngrok.asc | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null
!echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | sudo tee /etc/apt/sources.list.d/ngrok.list
!sudo apt-get update -y
!sudo apt-get install ngrok -y

In [ ]:
# TELECOM NOC PROCEDURAL RAG
!pip install -q datasets sentence-transformers chromadb pandas rank_bm25 transformers accelerate bitsandbytes

import torch
import chromadb
import pandas as pd
import string
import textwrap
import gc
import numpy as np
import os
import shutil
import json
import sqlite3
import re
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware Accelerator: {device}")

In [ ]:
# DB_PATH = "./telecom_vector_db"
# SQLITE_PATH = "telecom_sops.db"

import uuid

DB_PATH = f"/tmp/telecom_vector_db_{uuid.uuid4().hex}"
SQLITE_PATH = f"/tmp/telecom_sops_{uuid.uuid4().hex}.db"

# 1. Clean up old runs
if os.path.exists(DB_PATH): shutil.rmtree(DB_PATH)
if os.path.exists(SQLITE_PATH): os.remove(SQLITE_PATH)

# 2. Initialize SQLite (for Document Store)
conn = sqlite3.connect(SQLITE_PATH)
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS sops (
        sop_id TEXT PRIMARY KEY,
        vendor TEXT,
        severity TEXT,
        full_json TEXT
    )
''')
conn.commit()
print("SQLite initialized.")

# 3. Initialize ChromaDB (for Vector Store)
chroma_client = chromadb.PersistentClient(path=DB_PATH)
collection = chroma_client.get_or_create_collection(
    name="telecom_sops",
    metadata={"hnsw:space": "cosine"}
)
print("ChromaDB initialized.")

# 4. Load Embedding Model
print("🔹 Loading SOTA Embedding Model: BAAI/bge-base-en-v1.5")
embedding_func = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)

In [ ]:
print("Loading structured SOPs and rewriting for dense embeddings:")

with open("telecom_sops.json", "r") as f:
    dataset = json.load(f)

bm25_tokenized_corpus = []
batch_docs = []
batch_metadatas = []
batch_ids = []

seen_ids = set()

def simple_tokenize(text):
    return text.lower().translate(str.maketrans('', '', string.punctuation)).split()

for sop in tqdm(dataset, desc="Indexing"):
    sop_id = sop["sop_id"]

    if sop_id in seen_ids:
        continue
    seen_ids.add(sop_id)

    vendor = sop.get("vendor", "Unknown")
    severity = sop.get("severity", "UNKNOWN")

    # Conversational Data Representation
    term = sop.get('title', '').replace('SOP for ', '').replace(' Disruption', '')
    prose_content = f"This is a {severity} severity Standard Operating Procedure (SOP) for {vendor} equipment. "
    prose_content += f"It resolves disruptions caused by {term}. The procedure involves the following steps: "

    steps_text = " ".join([f"Step {s.get('step_number')}: {s.get('action')}. Execute command: `{s.get('command')}`." for s in sop.get('steps', [])])
    prose_content += steps_text

    # Override the original rigid search_content
    sop["search_content"] = prose_content

    # 1. Insert into SQLite (Full Document)
    cursor.execute(
        "INSERT INTO sops (sop_id, vendor, severity, full_json) VALUES (?, ?, ?, ?)",
        (sop_id, vendor, severity, json.dumps(sop))
    )

    # 2. Prepare for Vector DB & BM25
    bm25_tokenized_corpus.append(simple_tokenize(prose_content))
    batch_docs.append(prose_content)
    batch_ids.append(sop_id)
    batch_metadatas.append({"vendor": vendor, "severity": severity})

conn.commit()

# 3. Insert into ChromaDB
embeddings = embedding_func.encode(batch_docs, normalize_embeddings=True).tolist()
collection.add(
    documents=batch_docs,
    embeddings=embeddings,
    metadatas=batch_metadatas,
    ids=batch_ids
)

print(f"Indexed {collection.count()} SOPs with optimized prose embeddings.")

print("Building BM25 Index...")
bm25 = BM25Okapi(bm25_tokenized_corpus)
del bm25_tokenized_corpus
gc.collect()

In [ ]:
print("Loading SOTA Cross-Encoder: BAAI/bge-reranker-base")
reranker = CrossEncoder('BAAI/bge-reranker-base', device=device)

print("\nLoading Llama-3-8B (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "unsloth/llama-3-8b-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

llm_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=False,
    temperature=0.0
)

print("AI Models Loaded Successfully & Greedy Decoding Enforced.")

In [ ]:
!pip install -q gliner

In [ ]:
from gliner import GLiNER
import numpy as np
import json
import asyncio
from fastapi.responses import StreamingResponse
from transformers import TextIteratorStreamer
from threading import Thread
import time

# LOAD GLiNER (Zero-Shot Metadata Extractor)
print("Loading GLiNER for localized, zero-shot metadata extraction:")
# Load the model weights
ner_model = GLiNER.from_pretrained("urchade/gliner_small-v2.1")

ner_model = ner_model.to(device)
def extract_metadata_gliner(query):
    labels = ["telecom vendor", "severity level"]

    # Predict entities directly from the raw string
    entities = ner_model.predict_entities(query, labels)

    # Use a dictionary to guarantee the casing perfectly matches the generated JSON metadata
    VENDOR_MAP = {
        "cisco": "Cisco",
        "juniper": "Juniper",
        "nokia": "Nokia",
        "ericsson": "Ericsson",
        "huawei": "Huawei",
        "zte": "ZTE",
        "mavenir": "Mavenir",
        "samsung networks": "Samsung Networks",
        "samsung": "Samsung Networks", # Fallback if GLiNER just extracts 'Samsung'
        "ciena": "Ciena",
        "palo alto": "Palo Alto",
        "fortinet": "Fortinet",
        "check point": "Check Point",
        "f5": "F5"
    }

    filters = {}
    for ent in entities:
        text_val = ent["text"].lower()
        label = ent["label"]

        # Clean and normalize the extracted text to match ChromaDB metadata
        if label == "telecom vendor":
            for key, exact_vendor in VENDOR_MAP.items():
                if key in text_val:
                    filters["vendor"] = exact_vendor
                    break
        elif label == "severity level":
            for s in ["critical", "major", "minor", "warning"]:
                if s in text_val:
                    filters["severity"] = s.upper()
                    break

    return filters if filters else None

# THE OPTIMIZED QUERY PIPELINE
def process_query(query):
    # Extract Metadata Filters using Local GLiNER
    chroma_filters = extract_metadata_gliner(query)
    print(f"  [Diagnostics] Router applied filters: {chroma_filters}")

    # Dense Retrieval
    query_emb = embedding_func.encode([f"Represent this sentence for searching relevant passages: {query}"], normalize_embeddings=True).tolist()

    dense_res = collection.query(
        query_embeddings=query_emb,
        n_results=10,
        where=chroma_filters
    )

    if not dense_res['ids'][0]:
        return "No matching SOPs found for this specific hardware/fault.", []

    dense_hits = {id: score for id, score in zip(dense_res['ids'][0], dense_res['distances'][0])}

    # Sparse Retrieval (BM25)
    tokenized_query = simple_tokenize(query)
    sparse_scores = bm25.get_scores(tokenized_query)
    top_sparse_indices = np.argsort(sparse_scores)[-10:][::-1]

    # Tuned Reciprocal Rank Fusion (RRF) for dense datasets
    fusion_scores = {}
    k = 20 # Aggressively favor the top hits to reduce noise before the Cross-Encoder

    for rank, (doc_id, _) in enumerate(dense_hits.items()):
        fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

    for rank, idx in enumerate(top_sparse_indices):
        doc_id = batch_ids[idx]
        fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

    sorted_candidates = sorted(fusion_scores.items(), key=lambda x: x[1], reverse=True)[:5]
    top_ids = [doc_id for doc_id, _ in sorted_candidates]

    # Fetch FULL JSON from decoupled SQLite storage
    placeholders = ','.join(['?'] * len(top_ids))
    cursor.execute(f"SELECT sop_id, full_json FROM sops WHERE sop_id IN ({placeholders})", top_ids)
    rows = cursor.fetchall()

    id_to_json = {row[0]: json.loads(row[1]) for row in rows}
    candidate_docs = [id_to_json[uid] for uid in top_ids if uid in id_to_json]

    # Reranking via Cross-Encoder
    pairs = [[query, doc["search_content"]] for doc in candidate_docs]
    scores = reranker.predict(pairs)
    ranked_final = sorted(list(zip(candidate_docs, scores)), key=lambda x: x[1], reverse=True)

    # Expanded Context Window (Taking Top 3 instead of Top 2)
    final_sops = [doc for doc, score in ranked_final][:3]

    # Format Context for Operational Safety
    context_blocks = []
    for sop in final_sops:
        block = f"SOP ID: {sop['sop_id']} | Vendor: {sop.get('vendor')} | Severity: {sop.get('severity')}\n"
        block += f"Warnings: {', '.join(sop.get('safety_warnings', []))}\nSteps:\n"
        for step in sop.get('steps', []):
            block += f"  {step['step_number']}. {step['action']} -> Command: `{step['command']}`\n"
        context_blocks.append(block)

    context_string = "\n\n".join(context_blocks)

    # Strict Citation Prompting to prevent Hallucinations
    # Strict XML Chain-of-Thought Prompting
    # Robust Chain-of-Thought Prompting for 4-bit Models
    sys_prompt = (
        "You are a factual Tier-1 NOC AI Assistant. You receive multiple SOPs. "
        "You must follow these instructions exactly:\n"
        "1. Identify the SINGLE most relevant SOP for the user's issue.\n"
        "2. First, think step-by-step about why this SOP is correct. Prefix this section with 'REASONING:'.\n"
        "3. Second, provide your final procedural answer. Prefix this section with 'FINAL_ANSWER:'.\n"
        "4. In your final answer, do NOT generate artificial warnings unless the chosen SOP explicitly states them.\n"
        "5. In your final answer, append the exact SOP ID in brackets for EVERY CLI command (e.g., `clear ip bgp *` [SOP-123])."
    )

    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": f"Context SOPs:\n{context_string}\n\nUser Issue: {query}"},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # We remove max_length to avoid the Transformers conflict warning
    outputs = llm_pipe(prompt)

    full_out = outputs[0]["generated_text"]
    generated_text = full_out.split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()

    # Fail proof extraction: Split the string and take everything after the marker
    if "FINAL_ANSWER:" in generated_text:
        answer = generated_text.split("FINAL_ANSWER:")[-1].strip()
    else:
        # Emergency fallback if the model completely ignores formatting
        answer = generated_text

    return answer, final_sops

In [ ]:
def process_query_stream(query):
    """
    Streaming version of process_query that yields chunks as they're generated.
    """
    try:
        # Extract Metadata Filters using Local GLiNER
        chroma_filters = extract_metadata_gliner(query)
        print(f"  [Streaming] Router applied filters: {chroma_filters}")

        # Dense Retrieval
        query_emb = embedding_func.encode([f"Represent this sentence for searching relevant passages: {query}"], normalize_embeddings=True).tolist()

        dense_res = collection.query(
            query_embeddings=query_emb,
            n_results=10,
            where=chroma_filters
        )

        if not dense_res['ids'][0]:
            yield json.dumps({"type": "content", "content": "No matching SOPs found for this specific hardware/fault."})
            yield json.dumps({"type": "done"})
            return

        dense_hits = {id: score for id, score in zip(dense_res['ids'][0], dense_res['distances'][0])}

        # Sparse Retrieval (BM25)
        tokenized_query = simple_tokenize(query)
        sparse_scores = bm25.get_scores(tokenized_query)
        top_sparse_indices = np.argsort(sparse_scores)[-10:][::-1]

        # Reciprocal Rank Fusion
        fusion_scores = {}
        k = 20

        for rank, (doc_id, _) in enumerate(dense_hits.items()):
            fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

        for rank, idx in enumerate(top_sparse_indices):
            doc_id = batch_ids[idx]
            fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

        sorted_candidates = sorted(fusion_scores.items(), key=lambda x: x[1], reverse=True)[:5]
        top_ids = [doc_id for doc_id, _ in sorted_candidates]

        # Fetch FULL JSON from SQLite
        placeholders = ','.join(['?'] * len(top_ids))
        cursor.execute(f"SELECT sop_id, full_json FROM sops WHERE sop_id IN ({placeholders})", top_ids)
        rows = cursor.fetchall()

        id_to_json = {row[0]: json.loads(row[1]) for row in rows}
        candidate_docs = [id_to_json[uid] for uid in top_ids if uid in id_to_json]

        # Reranking via Cross-Encoder
        pairs = [[query, doc["search_content"]] for doc in candidate_docs]
        scores = reranker.predict(pairs)
        ranked_final = sorted(list(zip(candidate_docs, scores)), key=lambda x: x[1], reverse=True)

        final_sops = [doc for doc, score in ranked_final][:3]

        # Send sources first
        sources_data = []
        for sop in final_sops:
            sources_data.append({
                "sop_id": sop['sop_id'],
                "vendor": sop.get('vendor', 'Unknown'),
                "severity": sop.get('severity', 'UNKNOWN'),
                "title": sop.get('title', 'Untitled')
            })

        yield json.dumps({"type": "sources", "sources": sources_data})

        # Format Context for LLM
        context_blocks = []
        for sop in final_sops:
            block = f"SOP ID: {sop['sop_id']} | Vendor: {sop.get('vendor')} | Severity: {sop.get('severity')}\n"
            block += f"Warnings: {', '.join(sop.get('safety_warnings', []))}\nSteps:\n"
            for step in sop.get('steps', []):
                block += f"  {step['step_number']}. {step['action']} -> Command: `{step['command']}`\n"
            context_blocks.append(block)

        context_string = "\n\n".join(context_blocks)

        # Prompt Construction
        sys_prompt = (
            "You are a factual Tier-1 NOC AI Assistant. You receive multiple SOPs. "
            "You must follow these instructions exactly:\n"
            "1. Identify SINGLE most relevant SOP for user's issue.\n"
            "2. First, think step-by-step about why this SOP is correct. Prefix this section with 'REASONING:'.\n"
            "3. Second, provide your final procedural answer. Prefix this section with 'FINAL_ANSWER:'.\n"
            "4. In your final answer, do NOT generate artificial warnings unless chosen SOP explicitly states them.\n"
            "5. In your final answer, append the exact SOP ID in brackets for EVERY CLI command."
        )

        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": f"Context SOPs:\n{context_string}\n\nUser Issue: {query}"},
        ]

        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # Stream LLM generation
        inputs = tokenizer([prompt], return_tensors="pt").to(device)
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

        generation_kwargs = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=1024,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

        thread = Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()

        # Stream tokens as they are generated
        for new_text in streamer:
            yield json.dumps({"type": "content", "content": new_text})

        yield json.dumps({"type": "done"})

    except Exception as e:
        print(f"\nERROR IN STREAMING PROCESS_QUERY: {e}")
        yield json.dumps({"type": "error", "error": str(e)})

print("Streaming process_query function defined!")

In [ ]:
!pip install -qU langchain-google-genai
!pip install ragas

In [ ]:
import os
import pandas as pd
from tqdm.notebook import tqdm
from google.colab import userdata

# 1. Langchain Google GenAI Integrations
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# 2. Ragas v0.4+ Imports
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

# NEW TEST SET (Aligned with TRAP_CONCEPTS Matrix)
test_set = [
  {
    "question": "We are seeing massive 5G UE registration failures across the network. Subscribers can't even attach to the network. What function should I investigate?",
    "ground_truth": "Severity: CRITICAL. You must investigate the AMF (Access and Mobility Function) for 5G UE registration failures. STRICT WARNING: Do NOT confuse this with the SMF (Session Management Function). The SMF handles data sessions, but the core issue for initial attach and registration failures lies with the AMF."
  },
  {
    "question": "I need to run a live packet capture debug on the enterprise firewall to see why traffic is dropping. What command should I use? I need to make absolutely sure I don't drop the security policy.",
    "ground_truth": "Severity: CRITICAL. To safely debug firewall traffic, you must use a command like `fw control zdebug`. STRICT WARNING: Do absolutely NOT use `fw unloadlocal`. That command will instantly unload the firewall policy, leaving the device wide open to attack or completely isolating it."
  },
  {
    "question": "Mobile sessions are successfully establishing in the core, so signaling looks completely fine, but absolutely zero user data is actually flowing. Which protocol is failing?",
    "ground_truth": "Severity: CRITICAL. If signaling is active but user data is dropped, you must investigate GTP-U (GTP User Plane). STRICT WARNING: Do NOT confuse this with GTP-C (GTP Control Plane). Troubleshooting GTP-C will not resolve this, as the control plane sessions are already successfully established."
  },
  {
    "question": "Inter-cell handovers are failing between adjacent Nokia base stations, causing dropped calls when users drive, though the towers still have connectivity to the core. Which interface needs to be troubleshot?",
    "ground_truth": "Severity: CRITICAL. To resolve inter-cell handover failures between adjacent base stations, you must troubleshoot the X2 Interface. STRICT WARNING: Do NOT confuse this with the S1 Interface. Bouncing the S1 Interface will cause a total loss of core connectivity for all subscribers on those base stations."
  },
  {
    "question": "Our enterprise edge firewalls are experiencing a Split-Brain scenario where both devices assume the Active role. What is the impact of this?",
    "ground_truth": "Severity: CRITICAL. A Split-Brain HA scenario causes massive IP conflicts and ARP poisoning across the network. STRICT WARNING: Do NOT confuse this with an HA Sync Failure. A sync failure means configurations don't match, but a Split-Brain means both firewalls are actively fighting to route the same traffic."
  },
  {
    "question": "We have a critical fiber link failure on the Ciena transport ring. The optics team thinks it's a wavelength mismatch. Should we check the DWDM or CWDM configuration to avoid taking down the whole link?",
    "ground_truth": "Severity: CRITICAL. You must investigate the DWDM configuration for optical wavelength mismatches causing link failures. STRICT WARNING: Do NOT confuse this with CWDM, which pertains to reduced capacity and different spectrum spacing, but not this specific major optical failure."
  },
  {
    "question": "I need to drop all active data-plane IPsec VPN tunnels momentarily on the Check Point gateway. Should I use 'clear crypto ipsec sa' or 'clear crypto isakmp'?",
    "ground_truth": "Severity: MAJOR. To drop all active data-plane IPsec VPN tunnels momentarily, you must target `clear crypto ipsec sa`. STRICT WARNING: Do NOT confuse this with `clear crypto isakmp`. Running the wrong command will impact Phase 1 tunnels differently."
  },
  {
    "question": "We have major subscriber authentication failures in our LTE/5G interworking setup on the Samsung core. Is this an issue with the HSS or the UDM?",
    "ground_truth": "Severity: CRITICAL. For subscriber authentication failures in LTE/5G interworking, you must investigate the HSS. STRICT WARNING: Do NOT confuse this with the UDM. While the UDM handles 5G profiles, confusing the two will lead to subscriber profile inconsistency."
  },
  {
    "question": "There is a severe routing domain partition happening in the backbone on our ZTE routers. Do we need to look at OSPF Area 0 or the stub areas?",
    "ground_truth": "Severity: CRITICAL. To resolve a routing domain partition in the backbone, you must investigate OSPF Area 0. STRICT WARNING: Do NOT confuse this with a Non-Backbone Area or Stub Area. Misconfiguring this will further tear down backbone routing."
  },
  {
    "question": "Customers are complaining that their legitimate e-commerce traffic is being instantly dropped due to false positives on the F5 load balancer. Did someone put the WAF in Block Mode or Transparent Mode?",
    "ground_truth": "Severity: CRITICAL. If legitimate customer e-commerce traffic is being instantly dropped due to false positives, the WAF is likely in WAF Block Mode. STRICT WARNING: Do NOT confuse this with WAF Transparent Mode, which logs traffic without actively blocking it."
  }
]

# 3. SETUP MODELS & API via Langchain
api_key = userdata.get('GEMINI_API_KEY')
os.environ["GOOGLE_API_KEY"] = api_key

print("Initializing Gemma 3 27B and Embeddings...")

# Using Langchain to handle the proper .invoke() / .chat() translations for Ragas
evaluator_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(
        model="gemma-3-27b-it",
        temperature=0.0,
        max_retries=5
    )
)

# Text-embedding-004 for metric evaluations
evaluator_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
)

# 4. BUILD RAGAS DATASET
print("Running inferences and building the EvaluationDataset...")
samples = []

for item in tqdm(test_set):
    query = item["question"]
    gt = item["ground_truth"]

    # Execute local model generation pipeline
    ans, srcs = process_query(query)

    # Extract strings from the ChromaDB retrieved chunks
    retrieved_ctx = [doc["search_content"] for doc in srcs]

    # Creating the strict v0.4 SingleTurnSample object
    sample = SingleTurnSample(
        user_input=query,
        response=ans,
        retrieved_contexts=retrieved_ctx,
        reference=gt
    )
    samples.append(sample)

eval_dataset = EvaluationDataset(samples=samples)

# 5. CONFIGURE & RUN EVALUATION
# Rate limiting configuration to protect against API throttling
rate_limit_config = RunConfig(
    max_workers=2,
    max_retries=10,
    max_wait=30
)

print("\nEvaluating pipeline using Gemma 3 27B...")
results = evaluate(
    dataset=eval_dataset,
    metrics=[
        ContextPrecision(),
        ContextRecall(),
        Faithfulness(),
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=rate_limit_config
)

df_results = results.to_pandas()
display(df_results)

In [ ]:
import time
import torch
import pandas as pd
import numpy as np
from transformers import TextIteratorStreamer
from threading import Thread
import gc

def run_secure_noc_benchmark(test_queries):
    print(f"Starting Benchmark on {len(test_queries)} queries...")
    results = []

    # Force garbage collection and reset PyTorch VRAM trackers
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    for i, query in enumerate(test_queries):
        print(f"  -> Profiling Query {i+1}/{len(test_queries)}: {query[:40]}...")
        metrics = {"Query": query[:50] + "..."}

        # 1. RETRIEVAL PHASE (Routing, Dense, Sparse)
        t0 = time.time()

        chroma_filters = extract_metadata_gliner(query)
        query_emb = embedding_func.encode([f"Represent this sentence for searching relevant passages: {query}"], normalize_embeddings=True).tolist()

        dense_res = collection.query(query_embeddings=query_emb, n_results=10, where=chroma_filters)

        tokenized_query = simple_tokenize(query)
        sparse_scores = bm25.get_scores(tokenized_query)
        top_sparse_indices = np.argsort(sparse_scores)[-10:][::-1]

        # RRF logic
        fusion_scores = {}
        k = 20

        if dense_res['ids'] and dense_res['ids'][0]:
            dense_hits = {id: score for id, score in zip(dense_res['ids'][0], dense_res['distances'][0])}
            for rank, (doc_id, _) in enumerate(dense_hits.items()):
                fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

        for rank, idx in enumerate(top_sparse_indices):
            doc_id = batch_ids[idx]
            fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

        sorted_candidates = sorted(fusion_scores.items(), key=lambda x: x[1], reverse=True)[:5]
        top_ids = [doc_id for doc_id, _ in sorted_candidates]

        placeholders = ','.join(['?'] * len(top_ids))
        if top_ids:
            cursor.execute(f"SELECT sop_id, full_json FROM sops WHERE sop_id IN ({placeholders})", top_ids)
            rows = cursor.fetchall()
            id_to_json = {row[0]: json.loads(row[1]) for row in rows}
            candidate_docs = [id_to_json[uid] for uid in top_ids if uid in id_to_json]
        else:
            candidate_docs = []

        t1 = time.time()
        metrics["Retrieval_Time_sec"] = round(t1 - t0, 3)

        # 2. CROSS-ENCODER RERANKING PHASE
        t2 = time.time()
        if candidate_docs:
            pairs = [[query, doc["search_content"]] for doc in candidate_docs]
            scores = reranker.predict(pairs)
            ranked_final = sorted(list(zip(candidate_docs, scores)), key=lambda x: x[1], reverse=True)
            final_sops = [doc for doc, score in ranked_final][:3]
        else:
            final_sops = []
        t3 = time.time()
        metrics["Rerank_Time_sec"] = round(t3 - t2, 3)

        # Tracking SOP lengths
        avg_sop_len = np.mean([len(sop.get("search_content", "")) for sop in final_sops]) if final_sops else 0
        metrics["Avg_SOP_Length_chars"] = int(avg_sop_len)

        # 3. PROMPT CONSTRUCTION
        context_blocks = []
        for sop in final_sops:
            block = f"SOP ID: {sop['sop_id']} | Vendor: {sop.get('vendor')} | Severity: {sop.get('severity')}\n"
            block += f"Warnings: {', '.join(sop.get('safety_warnings', []))}\nSteps:\n"
            for step in sop.get('steps', []):
                block += f"  {step['step_number']}. {step['action']} -> Command: `{step['command']}`\n"
            context_blocks.append(block)

        context_string = "\n\n".join(context_blocks)

        sys_prompt = (
            "You are a factual Tier-1 NOC AI Assistant. You receive multiple SOPs. "
            "You must follow these instructions exactly:\n"
            "1. Identify the SINGLE most relevant SOP for the user's issue.\n"
            "2. First, think step-by-step about why this SOP is correct. Prefix this section with 'REASONING:'.\n"
            "3. Second, provide your final procedural answer. Prefix this section with 'FINAL_ANSWER:'.\n"
            "4. In your final answer, do NOT generate artificial warnings unless the chosen SOP explicitly states them.\n"
            "5. In your final answer, append the exact SOP ID in brackets for EVERY CLI command."
        )

        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": f"Context SOPs:\n{context_string}\n\nUser Issue: {query}"},
        ]

        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt_tokens = len(tokenizer.encode(prompt))
        metrics["Prompt_Tokens"] = prompt_tokens

        # 4. LLM GENERATION & TTFT PROFILING
        inputs = tokenizer([prompt], return_tensors="pt").to(device)
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

        generation_kwargs = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=1024,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

        t4 = time.time()
        thread = Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()

        generated_text = ""
        first_token_time = None

        # Consume the stream to measure TTFT accurately
        for new_text in streamer:
            if first_token_time is None:
                first_token_time = time.time()
            generated_text += new_text

        thread.join()
        t5 = time.time()

        ttft = first_token_time - t4 if first_token_time else 0
        gen_time = t5 - first_token_time if first_token_time else 0

        metrics["TTFT_sec"] = round(ttft, 3)
        metrics["Generation_Time_sec"] = round(gen_time, 3)

        gen_tokens = len(tokenizer.encode(generated_text))
        metrics["Generated_Tokens"] = gen_tokens
        metrics["Tokens_Per_Sec"] = round(gen_tokens / gen_time, 2) if gen_time > 0 else 0

        # 5. HARDWARE PROFILING
        peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
        metrics["Peak_VRAM_GB"] = round(peak_vram, 2)

        results.append(metrics)

    df_results = pd.DataFrame(results)
    return df_results

# Run the Benchmark
# Using a few questions from test set
test_queries = [
    "Urgent: I'm seeing stale routes on the US-East Core Cisco. I need to force a route refresh immediately but CANNOT drop active traffic. What do I type?",
    "We've got enterprise clients complaining about their IPsec VPN tunnels flapping. Can you pull the SOP to bounce the gateway service?",
    "Active mobile handovers are failing across the EU-Central RAN. Subscribers are dropping calls when driving. What gateway do I check?"
]

benchmark_df = run_secure_noc_benchmark(test_queries)
display(benchmark_df)

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio pydantic

In [ ]:
import nest_asyncio
import uvicorn
import threading
import traceback
import sqlite3
import shutil
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok, conf
from google.colab import userdata

# Applying nest_asyncio so Uvicorn plays nicely with Colab's event loop
nest_asyncio.apply()

SQLITE_PATH = "app.db"
def process_query(query):
    return "answer", [{"sop_id": "1", "vendor": "V", "severity": "S", "title": "T"}]

conn = sqlite3.connect(SQLITE_PATH, check_same_thread=False)
cursor = conn.cursor()

# 1. Define the API Schema
class QueryRequest(BaseModel):
    query: str

class SopResponse(BaseModel):
    sop_id: str
    vendor: str
    severity: str
    title: str

class QueryResponse(BaseModel):
    answer: str
    retrieved_sops: list[SopResponse]

# 2. Initialize FastAPI
app = FastAPI(title="NetRestore RAG API", version="1.0")

# 3. Define the POST Endpoint
@app.post("/ask", response_model=QueryResponse)
async def ask_netrestore(request: QueryRequest):
    try:
        print(f"\nReceived query from frontend: {request.query}")

        # Call pipeline
        ans, srcs = process_query(request.query)

        print("Query processed successfully! Sending back to frontend...")

        # Format the sources cleanly for the JSON response
        formatted_srcs = []
        for s in srcs:
            formatted_srcs.append(SopResponse(
                sop_id=s.get("sop_id", "N/A"),
                vendor=s.get("vendor", "Unknown"),
                severity=s.get("severity", "UNKNOWN"),
                title=s.get("title", "Untitled")
            ))

        return QueryResponse(answer=ans, retrieved_sops=formatted_srcs)

    except Exception as e:
        print("\nERROR IN PROCESS_QUERY:")
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


# 4. Expose server to the internet

ngrok_exec = shutil.which("ngrok")
if not ngrok_exec:
    raise RuntimeError("Ngrok binary not found!")

pyngrok_config = conf.PyngrokConfig(ngrok_path=ngrok_exec)
conf.set_default(pyngrok_config)

ngrok_token = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(ngrok_token, pyngrok_config=pyngrok_config)

# Close any existing tunnels to prevent errors
ngrok.kill()

public_url = ngrok.connect(8501, pyngrok_config=pyngrok_config).public_url

print(f"NETRESTORE API IS LIVE AT: {public_url}/ask")

# 5. Start the Server in a Background Thread
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8501)

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()

print("Background thread started. API is ready to receive requests!")

In [ ]:
# Adding streaming endpoint to existing FastAPI app
@app.post("/ask-stream")
async def ask_netrestore_stream(request: QueryRequest):
    async def generate_stream():
        try:
            print(f"\n[Streaming] Received query: {request.query}")

            # Generate streaming response
            for chunk in process_query_stream(request.query):
                yield f"data: {chunk}\n\n"

        except Exception as e:
            print(f"\n[Streaming] Error: {e}")
            error_chunk = json.dumps({"type": "error", "error": str(e)})
            yield f"data: {error_chunk}\n\n"

    return StreamingResponse(
        generate_stream(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "Connection": "keep-alive",
            "Access-Control-Allow-Origin": "*"
        }
    )